# Assignment #1: The Donut Effect for Philadelphia ZIP Codes

In this assignment, we will practice our `pandas` skills and explore the ["Donut Effect"](https://www.gsb.stanford.edu/faculty-research/publications/donut-effect-how-covid-19-shapes-real-estate) within Philadelphia. The "Donut Effect" describes the following phenomenon: with more flexible working options and pandemic-driven density fears, people left urban dense cores and opted for more space in city suburbs, driving home and rental prices up in the suburbs relative to city centers.

We will be working with [Zillow data](https://www.zillow.com/research/data/) for the Zillow Home Value Index (ZHVI) for Philadelphia ZIP codes. The goal will be to calculate home price appreciation in Philadelphia, comparing those ZIP codes in Center City (the central business district) to those not in Center City.

## 1. Load the data

I've already downloaded the relevant data file and put in the `data/` folder. Let's load it using `pandas`.  

**Note:** Be sure to use a *relative* file path to make it easier to load your data when grading. See [this guide](https://musa-550-fall-2023.github.io/resource/file-paths.html) for more info.

In [29]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


df = pd.read_csv('data/Zip_zhvi_uc_sfrcondo_tier_0.33_0.67_sm_sa_month.csv')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26269 entries, 0 to 26268
Columns: 328 entries, RegionID to 2026-07-31
dtypes: float64(319), int64(3), object(6)
memory usage: 65.7+ MB


## 2. Trim the data to just Philadelphia

Select the subset of the dataframe for Philadelphia, PA.

In [30]:
df_philly = df[(df['City'] == 'Philadelphia') & (df['State'] == 'PA')]
df_philly.columns

Index(['RegionID', 'SizeRank', 'RegionName', 'RegionType', 'StateName',
       'State', 'City', 'Metro', 'CountyName', '2000-01-31',
       ...
       '2025-10-31', '2025-11-30', '2025-12-31', '2026-01-31', '2026-02-28',
       '2026-03-31', '2026-04-30', '2026-05-31', '2026-06-30', '2026-07-31'],
      dtype='object', length=328)

## 3. Melt the data into tidy format

Let's transform the data from wide to tidy using the `pd.melt()` function. Create a new column in your data called "ZHVI" that holds the ZHVI values.

In [31]:
df_melt = pd.melt(
    df_philly, 
    id_vars=['RegionID', 'SizeRank', 'RegionName', 'RegionType', 'StateName','State', 'City', 'Metro', 'CountyName'], 
    var_name='Date', 
    value_name='ZHVI')

In [32]:
df_melt.head()

,RegionID,SizeRank,RegionName,RegionType,StateName,State,City,Metro,CountyName,Date,ZHVI
0,65787,167,19120,zip,PA,PA,Philadelphia,"Philadelphia-Camden-Wilmington, PA-NJ-DE-MD",Philadelphia County,2000-01-31,51789.832936
1,65791,225,19124,zip,PA,PA,Philadelphia,"Philadelphia-Camden-Wilmington, PA-NJ-DE-MD",Philadelphia County,2000-01-31,40495.692372
2,65779,266,19111,zip,PA,PA,Philadelphia,"Philadelphia-Camden-Wilmington, PA-NJ-DE-MD",Philadelphia County,2000-01-31,79753.120073
3,65810,327,19143,zip,PA,PA,Philadelphia,"Philadelphia-Camden-Wilmington, PA-NJ-DE-MD",Philadelphia County,2000-01-31,46552.041802
4,65816,421,19149,zip,PA,PA,Philadelphia,"Philadelphia-Camden-Wilmington, PA-NJ-DE-MD",Philadelphia County,2000-01-31,59028.318097


## 4. Split the data for ZIP codes in/outside Center City

To compare home appreciation in Center City vs. outside Center City, we'll need to split the data into two dataframes, one that holds the Center City ZIP codes and one that holds the data for the rest of the ZIP codes in Philadelphia.

To help with this process, I've included a list of ZIP codes that make up the "greater Center City" region of Philadelphia. Use this list to split the melted data into two dataframes.

In [33]:
greater_center_city_zip_codes = [
    19123,
    19102,
    19103,
    19106,
    19107,
    19109,
    19130,
    19146,
    19147,
]

In [34]:
df_in_center = df_melt[df_melt["RegionName"].isin(greater_center_city_zip_codes)]
df_out_center = df_melt[~df_melt["RegionName"].isin(greater_center_city_zip_codes)]

## 5. Compare home value appreciation in Philadelpia

In this step, we'll calculate the average percent increase in ZHVI from March 2020 to March 2022 for ZIP codes in/out of Center City. We'll do this by:

- Writing a function (see the template below) that will calculate the percent increase in ZHVI from March 31, 2020 to March 31, 2022
- Group your data and apply this function to calculate the ZHVI percent change for each ZIP code in Philadelphia. Do this for both of your dataframes from the previous step.
- Calculate the average value across ZIP codes for both sets of ZIP codes and then compare

You should see much larger growth for ZIP codes outside of Center City...the Donut Effect! 

In [35]:
def calculate_percent_increase(group_df):
    """
    Calculate the percent increase from 2020-03-31 to 2022-03-31.
    
    Note that `group_df` is the DataFrame for each group.
    """
    

    
    origin_value = group_df[group_df['Date'] == '2020-03-31']['ZHVI'].values[0]
    final_value = group_df[group_df['Date'] == '2022-03-31']['ZHVI'].values[0]

    percent_increase = ((final_value - origin_value) / origin_value) * 100

    return percent_increase

In [36]:
mean_df_in_center = df_in_center.groupby("RegionName").apply(calculate_percent_increase).mean()
mean_df_out_center = df_out_center.groupby("RegionName").apply(calculate_percent_increase).mean()

print(f"average value within city center is {mean_df_in_center:.2f}, average value outside city center is {mean_df_out_center:.2f}, and their difference is {(mean_df_in_center - mean_df_out_center):.2f}")

average value within city center is 6.08, average value outside city center is 18.07, and their difference is -12.00
